# 15 – Daten für das Lagebild

**Ziel:** zwei Tabellen, aus denen alles Weitere gerechnet wird:

1. **Meldedaten:** Influenza je Kreis und Woche für alle 400 Kreise: Fälle, Einwohner, Inzidenz, Saison und Zeitachse relativ zum Karneval.
2. **Abwasser:** Influenza A+B im Abwasser je Kreis und Woche (AMELAG).

**Entscheidungen:**

- **Alle 400 Kreise.** Das Lagebild ordnet jeden Kreis ein. Die Gruppen der Voranalyse (5 Karnevalsstädte, 7 Kontrollstädte, Nachbarkreise) bleiben als Spalte `Raum` erhalten. Sie dienen nur der Darstellung und dem Prüffall Karneval (Notebook 17); das Modell nutzt sie nicht.
- **Nur Influenza.** Für COVID-19 war die Prognose im Test in keiner Woche besser als „bleibt wie zuletzt“, weil sich das Testverhalten 2023/24 geändert hat (Fassung mit COVID-19 in `99_Archiv/Fassung Influenza+COVID (16.11.2026)`).
- **Kein Wetter, keine Schulferien.** Wetter gibt es nur für 12 Städte und brachte im Benchmark der Voranalyse keinen Nutzen. Schulferien änderten den Prognosefehler um höchstens 0,004 (geprüft am 16.11.2026). Beide fallen weg, damit das Lagebild nur zwei Quellen braucht.

**Eingang:** `meldedaten_inzidenz.csv` (Notebook 03), `umland_kreise.csv`, `karnevalstermine.csv` (Notebook 02), `02_Daten/AMELAG/amelag_einzelstandorte.tsv`
**Ergebnis:** `lagebild_daten.csv`, `abwasser_kreis_woche.csv`

In [ ]:
from pathlib import Path
import pandas as pd

from lagebild import abwasser_wochen

ORDNER = Path("../02_Daten/aufbereitet")
STAEDTE = {"SK Köln": "Köln", "SK Düsseldorf": "Düsseldorf", "SK Bonn": "Bonn", "SK Mainz": "Mainz", "StädteRegion Aachen": "Aachen"}
KONTROLLEN = ["SK Berlin", "SK Bremen", "Region Hannover", "SK Bielefeld", "SK Kiel", "SK Lübeck", "SK Kassel"]
LAND = {"01": "SH", "02": "HH", "03": "NI", "04": "HB", "05": "NW", "06": "HE", "07": "RP", "08": "BW",
        "09": "BY", "10": "SL", "11": "BE", "12": "BB", "13": "MV", "14": "SN", "15": "ST", "16": "TH"}

meldungen = pd.read_csv(ORDNER / "meldedaten_inzidenz.csv", dtype={"AGS": str}, parse_dates=["Wochenbeginn"])
meldungen["AGS"] = meldungen["AGS"].str.zfill(5)
print(meldungen["Kreis"].nunique(), "Kreise | Wochen bis", meldungen["Wochenbeginn"].max().date())

## Schritt 1: Influenza auswählen

`meldedaten_inzidenz.csv` enthält nur noch Influenza; die Auswahl sichert das ab. Wochen ohne Meldung sind dort schon mit 0 Fällen enthalten.

In [ ]:
d = meldungen[meldungen["Erreger"] == "Influenza"].drop(columns="Erreger").copy()
assert len(d) == d["Kreis"].nunique() * d["Wochenbeginn"].nunique(), "Nicht jeder Kreis hat jede Woche"
d.groupby("Meldejahr")["Faelle"].sum().rename("Influenza-Fälle").to_frame().T

## Schritt 2: Raum und Land

- **Raum:** `umland_kreise.csv` (aus den BKG-Kreisgrenzen) enthält die Nachbarkreise (Ring 1) und übernächsten Kreise (Ring 2) der Karnevalsstädte. Alle anderen Kreise heißen „Übriger Kreis“.
- **Land:** die ersten zwei Stellen des Kreisschlüssels (für die Länderebene, Notebook 20).

In [ ]:
umland = pd.read_csv(ORDNER / "umland_kreise.csv", dtype={"AGS": str})
umland["AGS"] = umland["AGS"].str.zfill(5)
d = d.merge(umland[["AGS", "Ring"]], on="AGS", how="left")
d["Raum"] = d.pop("Ring").map({1: "Nachbarkreis", 2: "Übernächster Kreis"}).fillna("Übriger Kreis")
d.loc[d["Kreis"].isin(STAEDTE), "Raum"] = "Karnevalsstadt"
d.loc[d["Kreis"].isin(KONTROLLEN), "Raum"] = "Kontrollstadt"
d["Land"] = d["AGS"].str[:2].map(LAND)

assert d["Land"].notna().all()
d.drop_duplicates("Kreis")["Raum"].value_counts()

## Schritt 3: Saison und Zeitachse

- **Saison:** Kalenderwoche 40 bis 39 des Folgejahres, benannt nach dem Jahr, in dem sie endet (Saison 2025 = Oktober 2024 bis September 2025). So liegt jede Grippewelle in einer Saison.
- **Woche_rel:** Wochen relativ zur Woche von Weiberfastnacht (0 = Woche von Weiberfastnacht). Nur für den Prüffall Karneval.

In [ ]:
termine = pd.read_csv(ORDNER / "karnevalstermine.csv", parse_dates=["Weiberfastnacht"])
termine["Montag_WF"] = termine["Weiberfastnacht"] - pd.to_timedelta(termine["Weiberfastnacht"].dt.weekday, unit="D")
d["Saison"] = d["Meldejahr"] + (d["Meldewoche"] >= 40).astype(int)
d = d.merge(termine[["Jahr", "Montag_WF"]], left_on="Saison", right_on="Jahr", how="left").drop(columns="Jahr")
d["Woche_rel"] = (d["Wochenbeginn"] - d.pop("Montag_WF")).dt.days // 7

assert d["Woche_rel"].notna().all(), "karnevalstermine.csv reicht nicht bis zur laufenden Saison (Notebook 02)"
d[d["Woche_rel"] == 0].drop_duplicates("Saison")[["Saison", "Wochenbeginn"]].set_index("Saison").T

## Schritt 4: Abwasser je Kreis und Woche

- **Influenza A+B** aus AMELAG, gemessen seit Oktober 2023, also erst ab Saison 2024.
- **Wochenwert:** Mittel der logarithmierten, nach Zufluss normalisierten Viruslast aller Messungen eines Kreises in der Woche. Mehrere Klärwerke eines Kreises werden gemittelt.
- **Zuordnung** über den Standortnamen; bei gleichem Namen gilt die kreisfreie Stadt. Ein Klärwerk deckt seinen Kreis oft nur teilweise ab (z. B. Aachen für die StädteRegion). Klärwerke, deren Name keinem Kreis entspricht, entfallen.

In [ ]:
abw, messungen = abwasser_wochen("../02_Daten/AMELAG/amelag_einzelstandorte.tsv", sorted(d["Kreis"].unique()))
assert {"SK Köln", "SK Mainz", "StädteRegion Aachen"} <= set(abw["Kreis"])      # Düsseldorf und Bonn messen Influenza nicht
print(abw["Kreis"].nunique(), "Kreise mit Messstelle | Messungen von", messungen["datum"].min().date(), "bis", messungen["datum"].max().date())

## Schritt 5: Speichern

| Spalte | Bedeutung |
|---|---|
| AGS, Kreis, Land | Kreisschlüssel, Name, Bundesland |
| Raum | Karnevalsstadt, Kontrollstadt, Nachbarkreis, Übernächster Kreis oder Übriger Kreis |
| Meldejahr, Meldewoche, Wochenbeginn | Kalenderwoche (ISO) und ihr Montag |
| Saison, Woche_rel | Saison (Oktober bis September) und Wochen relativ zu Weiberfastnacht |
| Faelle, Einwohner, Inzidenz | gemeldete Fälle, Einwohner, Fälle je 100.000 und Woche |

`abwasser_kreis_woche.csv`: Kreis, Wochenbeginn, log_vl (logarithmierte Viruslast).

In [ ]:
spalten = ["AGS", "Kreis", "Land", "Raum", "Meldejahr", "Meldewoche", "Wochenbeginn", "Saison", "Woche_rel", "Faelle", "Einwohner", "Inzidenz"]
d = d[spalten].sort_values(["Kreis", "Wochenbeginn"])
assert d["Kreis"].nunique() == 400 and not d.duplicated(["Kreis", "Wochenbeginn"]).any()
d.to_csv(ORDNER / "lagebild_daten.csv", index=False)
abw.to_csv(ORDNER / "abwasser_kreis_woche.csv", index=False)
print("Gespeichert:", len(d), "Zeilen, Wochen bis", d["Wochenbeginn"].max().date(), "| Abwasser:", len(abw), "Kreis-Wochen")

## Was du für die Methodik mitnehmen kannst

- **Untersuchungsraum:** alle 400 Kreise, Influenza, Wochen ab 2015.
- **Zwei Quellen:** Meldedaten (RKI SurvStat) und Abwasser (RKI AMELAG, Influenza A+B, seit Oktober 2023, 74 Kreise mit Messstelle).
- **Saison:** Kalenderwoche 40 bis 39, benannt nach dem Jahr, in dem sie endet.
- **Weggelassen:** Wetter (nur 12 Stationen, ohne Nutzen) und Schulferien (ohne messbaren Nutzen).